In [5]:
import os, glob, pathlib, getpass, time

def set_scratch_env():
    # 1) If already set, keep it
    scr = os.environ.get("SCRATCH", "").strip()
    if scr and os.path.isdir(scr):
        return scr

    # 2) Try SLURM envs (if any)
    user = os.environ.get("USER", getpass.getuser())
    jid  = os.environ.get("SLURM_JOB_ID", "").strip()
    if user and jid:
        cand = f"/scratch/{user}/{jid}"
        if os.path.isdir(cand):
            os.environ["SCRATCH"] = cand
            return cand

    # 3) Fall back: pick the most recent /scratch/<user>/* directory
    base = f"/scratch/{user}"
    paths = sorted(glob.glob(f"{base}/*"), key=lambda p: os.path.getmtime(p), reverse=True)
    for p in paths:
        try:
            test = pathlib.Path(p) / ".__write_test__"
            with open(test, "w") as f: f.write("ok")
            test.unlink()
            os.environ["SCRATCH"] = p
            return p
        except Exception:
            continue

    raise RuntimeError("Could not locate a writable /scratch/<user>/<jobid>. "
                       "Restart the kernel inside the Slurm job or choose another kernel.")

SCRATCH = set_scratch_env()
# print("SCRATCH =", SCRATCH)

In [6]:
import os, pathlib, subprocess

SCRATCH = os.environ["SCRATCH"]  # you already have it now
RUN_ROOT = os.path.join(SCRATCH, "hf_runs")
HF_HOME = os.path.join(RUN_ROOT, "hf_home")
TRANSFORMERS_CACHE = os.path.join(RUN_ROOT, "transformers_cache")
TORCH_HOME = os.path.join(RUN_ROOT, "torch_home")
for d in [RUN_ROOT, HF_HOME, TRANSFORMERS_CACHE, TORCH_HOME]:
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = HF_HOME
os.environ["TRANSFORMERS_CACHE"] = TRANSFORMERS_CACHE
os.environ["TORCH_HOME"] = TORCH_HOME

# print("SCRATCH =", SCRATCH)
# print("RUN_ROOT =", RUN_ROOT)

In [8]:
import os, pathlib, subprocess

# Get SCRATCH from the kernel env; if missing, read from login shell
scratch = os.environ.get("SCRATCH")
if not scratch:
    scratch = subprocess.run(["bash","-lc","echo -n $SCRATCH"], capture_output=True, text=True).stdout
if not scratch:
    raise RuntimeError("SCRATCH is not set in this kernel. Start the notebook via an HPCC allocation (OnDemand).")

run_root = f"{scratch}/hf_runs"
env = {
    "HF_HOME":               f"{run_root}/hf_home",
    "HUGGINGFACE_HUB_CACHE": f"{run_root}/hf_home/hub",
    "TRANSFORMERS_CACHE":    f"{run_root}/transformers",
    "TORCH_HOME":            f"{run_root}/torch_home",
    "PIP_CACHE_DIR":         f"{run_root}/pip_cache",
}
os.environ.update(env)

# Make sure the dirs exist
for p in env.values():
    pathlib.Path(p).mkdir(parents=True, exist_ok=True)

# print("SCRATCH =", scratch)
for k, v in env.items():
    pass
    # print(f"{k} = {v}")

In [10]:
import os, subprocess, textwrap
# print({k: os.environ[k] for k in ["HF_HOME","HUGGINGFACE_HUB_CACHE","TRANSFORMERS_CACHE","TORCH_HOME","PIP_CACHE_DIR"]})
# print(subprocess.run(["bash","-lc","set -o pipefail; df -h \"$SCRATCH\" | sed -n '1,2p'"], text=True, capture_output=True).stdout)

In [11]:
! nvidia-smi

Sun Oct 26 22:16:50 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.172.08             Driver Version: 570.172.08     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          On  |   00000000:10:00.0 Off |                    0 |
| N/A   33C    P0             66W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [12]:
! df -h

Filesystem            Size  Used Avail Use% Mounted on
devtmpfs              504G     0  504G   0% /dev
tmpfs                 504G  960K  504G   1% /dev/shm
tmpfs                 504G  178M  504G   1% /run
tmpfs                 504G     0  504G   0% /sys/fs/cgroup
/dev/mapper/vg1-root   32G   11G   22G  33% /
/dev/nvme0n1p2       1016M  218M  799M  22% /boot
/dev/nvme0n1p6        1.5T   11G  1.5T   1% /scratch
/dev/nvme0n1p1        599M  5.9M  594M   1% /boot/efi
/dev/mapper/vg1-var   8.0G  788M  7.3G  10% /var
/dev/mapper/vg1-tmp   100G  755M  100G   1% /tmp
home                  131T   19T  113T  15% /rhome
bigdata               3.8P  3.1P  693T  83% /bigdata


In [23]:
%pip install beir=='2.0.0'

Note: you may need to restart the kernel to use updated packages.


In [25]:
%pip install InstructorEmbedding

Note: you may need to restart the kernel to use updated packages.


In [26]:
%pip install sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [15]:
import torch
dev = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(dev)

/rhome/znazi002/.conda/envs/pytorch_env/lib/python3.13/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


cuda


In [16]:
torch.__version__

'2.6.0+cu124'

In [17]:
import transformers

/rhome/znazi002/.conda/envs/pytorch_env/lib/python3.13/site-packages/transformers/utils/hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [18]:
transformers.__version__

'4.55.4'

In [19]:
import accelerate

In [20]:
accelerate.__version__

'1.8.1'

In [21]:
import torch
print("torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision("high")
DTYPE = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8) else torch.float16
print("Using dtype:", DTYPE)

torch: 2.6.0+cu124
CUDA: True
GPU: NVIDIA A100-SXM4-80GB
Using dtype: torch.bfloat16


In [27]:
import torchvision

In [28]:
import os
import pathlib
import logging
from datetime import timedelta
from typing import List, Dict, Union, Tuple

import numpy as np
import torch
from torch import Tensor
import torch.distributed as dist
from tqdm import trange
from transformers import AutoTokenizer, AutoModel
from transformers.file_utils import PaddingStrategy

from beir import util, LoggingHandler
from beir.retrieval import models
from beir.datasets.data_loader import GenericDataLoader
from beir.retrieval.evaluation import EvaluateRetrieval
from beir.retrieval.search.dense import DenseRetrievalExactSearch as DRES

import torch
import numpy as np
from typing import List, Dict
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel
from beir.retrieval.search.dense import DenseRetrievalExactSearch as DRES
from sentence_transformers import SentenceTransformer

# Configure logging
logging.basicConfig(format='%(asctime)s - %(message)s',
                    datefmt='%Y-%m-%d %H:%M:%S',
                    level=logging.INFO,
                    handlers=[LoggingHandler()])
logger = logging.getLogger(__name__)


class InstructorDenseRetriever:
    def __init__(self, model_path="hkunlp/instructor-large"):
        """Initialize the Instructor Embedding model for both query and document encoding"""
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # Load Model
        self.model = SentenceTransformer(model_path, device=self.device)

    def encode_queries(self, queries: List[str], batch_size: int = 64, **kwargs) -> np.ndarray:
        """Encodes queries into dense embeddings using instructional prompts"""
        instruction = "Represent the query for retrieving supporting documents:"
        query_embeddings = []

        with torch.no_grad():
            for start_idx in trange(0, len(queries), batch_size):
                batch_queries = queries[start_idx:start_idx+batch_size]
                embeddings = self.model.encode(batch_queries, batch_size=batch_size, convert_to_numpy=True, prompt=instruction)
                query_embeddings.append(embeddings)

        return np.vstack(query_embeddings)

    def encode_corpus(self, corpus: List[Dict[str, str]], batch_size: int = 64, **kwargs) -> np.ndarray:
        """Encodes documents into dense embeddings using instructional prompts"""
        instruction = "Represent the document for retrieval:"
        corpus_embeddings = []

        with torch.no_grad():
            for start_idx in trange(0, len(corpus), batch_size):
                batch_corpus = [row.get('title', '') + " " + row.get('text', '') for row in corpus[start_idx:start_idx+batch_size]]
                embeddings = self.model.encode(batch_corpus, batch_size=batch_size, convert_to_numpy=True, prompt=instruction)
                corpus_embeddings.append(embeddings)

        return np.vstack(corpus_embeddings)

In [29]:
instructor = DRES(InstructorDenseRetriever())
retriever_instructor = EvaluateRetrieval(instructor, score_function="cos_sim") # or "cos_sim" for cosine similarity

2025-10-26 22:27:17 - Load pretrained SentenceTransformer: hkunlp/instructor-large


modules.json:   0%|          | 0.00/461 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

2_Dense/pytorch_model.bin:   0%|          | 0.00/3.15M [00:00<?, ?B/s]

In [30]:
# Download and load dataset
dataset = "nfcorpus" # dataset name
url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{dataset}.zip"
out_dir = "datasets"
data_path = util.download_and_unzip(url, out_dir)

2025-10-26 22:27:40 - Downloading nfcorpus.zip ...


datasets/nfcorpus.zip:   0%|          | 0.00/2.34M [00:00<?, ?iB/s]

2025-10-26 22:27:50 - Unzipping nfcorpus.zip ...


In [31]:
# Load corpus, queries, and qrels
corpus, queries, qrels = GenericDataLoader(data_folder=data_path).load(split="test")

2025-10-26 22:36:11 - Loading Corpus...


  0%|          | 0/3633 [00:00<?, ?it/s]

2025-10-26 22:36:11 - Loaded 3633 TEST Documents.
2025-10-26 22:36:11 - Doc Example: {'text': 'Recent studies have suggested that statins, an established drug group in the prevention of cardiovascular mortality, could delay or prevent breast cancer recurrence but the effect on disease-specific mortality remains unclear. We evaluated risk of breast cancer death among statin users in a population-based cohort of breast cancer patients. The study cohort included all newly diagnosed breast cancer patients in Finland during 1995–2003 (31,236 cases), identified from the Finnish Cancer Registry. Information on statin use before and after the diagnosis was obtained from a national prescription database. We used the Cox proportional hazards regression method to estimate mortality among statin users with statin use as time-dependent variable. A total of 4,151 participants had used statins. During the median follow-up of 3.25 years after the diagnosis (range 0.08–9.0 years) 6,011 participants die

In [32]:
import json
# load paraphrased dataset
with open(f"{dataset}_query_paraphrased_gpt4o.json", encoding='utf-8') as f:
    # Load the JSON data into a Python dictionary
    queries_para = json.load(f)

In [33]:
queries_p = {}
for q in queries_para:
  queries_p[q] = queries_para[q]['query_p']

In [34]:
results_instructor = retriever_instructor.retrieve(corpus, queries_p)

2025-10-26 22:36:44 - Encoding Queries...


  0%|          | 0/3 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

 33%|███▎      | 1/3 [00:01<00:02,  1.44s/it]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

 93%|█████████▎| 27/29 [00:19<00:01,  1.93it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

 97%|█████████▋| 28/29 [00:20<00:00,  2.25it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 29/29 [00:20<00:00,  1.43it/s]


In [35]:
# Evaluate your model with NDCG@k, MAP@K, Recall@K and Precision@K where k = [1,3,5,10,100,1000]
ndcg, _map, recall, precision = retriever_instructor.evaluate(qrels, results_instructor, retriever_instructor.k_values)

2025-10-26 22:38:12 - For evaluation, we ignore identical query and document ids (default), please explicitly set ``ignore_identical_ids=False`` to ignore this.
2025-10-26 22:38:12 - 

2025-10-26 22:38:12 - NDCG@1: 0.3669
2025-10-26 22:38:12 - NDCG@3: 0.3288
2025-10-26 22:38:12 - NDCG@5: 0.3194
2025-10-26 22:38:12 - NDCG@10: 0.2904
2025-10-26 22:38:12 - NDCG@100: 0.2819
2025-10-26 22:38:12 - NDCG@1000: 0.3740
2025-10-26 22:38:12 - 

2025-10-26 22:38:12 - MAP@1: 0.0411
2025-10-26 22:38:12 - MAP@3: 0.0668
2025-10-26 22:38:12 - MAP@5: 0.0808
2025-10-26 22:38:12 - MAP@10: 0.0955
2025-10-26 22:38:12 - MAP@100: 0.1287
2025-10-26 22:38:12 - MAP@1000: 0.1438
2025-10-26 22:38:12 - 

2025-10-26 22:38:12 - Recall@1: 0.0411
2025-10-26 22:38:12 - Recall@3: 0.0789
2025-10-26 22:38:12 - Recall@5: 0.1021
2025-10-26 22:38:12 - Recall@10: 0.1386
2025-10-26 22:38:12 - Recall@100: 0.3078
2025-10-26 22:38:12 - Recall@1000: 0.6407
2025-10-26 22:38:12 - 

2025-10-26 22:38:12 - P@1: 0.3839
2025-10-26 22:38:12

In [36]:
print(f"Model: Instructor; Dataset: {dataset} (paraphrased)")
print("-" * 150)
print(ndcg)
print(_map)
print(recall)
print(precision)

Model: Instructor; Dataset: nfcorpus (paraphrased)
------------------------------------------------------------------------------------------------------------------------------------------------------
{'NDCG@1': 0.36687, 'NDCG@3': 0.32884, 'NDCG@5': 0.3194, 'NDCG@10': 0.29042, 'NDCG@100': 0.28188, 'NDCG@1000': 0.37396}
{'MAP@1': 0.04105, 'MAP@3': 0.06679, 'MAP@5': 0.08082, 'MAP@10': 0.09555, 'MAP@100': 0.12872, 'MAP@1000': 0.14383}
{'Recall@1': 0.04105, 'Recall@3': 0.07888, 'Recall@5': 0.10214, 'Recall@10': 0.13857, 'Recall@100': 0.30784, 'Recall@1000': 0.64071}
{'P@1': 0.3839, 'P@3': 0.31579, 'P@5': 0.2904, 'P@10': 0.22322, 'P@100': 0.07901, 'P@1000': 0.02118}


In [46]:
import json
import pytrec_eval

# Use pytrec_eval to get per-query metrics
k_values = [10]
map_string = "map_cut." + ",".join([str(k) for k in k_values])
ndcg_string = "ndcg_cut." + ",".join([str(k) for k in k_values])
recall_string = "recall." + ",".join([str(k) for k in k_values])
precision_string = "P." + ",".join([str(k) for k in k_values])

evaluator = pytrec_eval.RelevanceEvaluator(qrels, {map_string, ndcg_string, recall_string, precision_string})
scores = evaluator.evaluate(results_instructor)

# Extract per-query metrics
per_query_metrics = {}
for query_id in scores.keys():
    per_query_metrics[query_id] = {
        "ndcg@10": scores[query_id]["ndcg_cut_10"],
        "recall@10": scores[query_id]["recall_10"],
        "map@10": scores[query_id]["map_cut_10"]
    }

# Calculate average metrics
avg_ndcg_10 = sum([m["ndcg@10"] for m in per_query_metrics.values()]) / len(per_query_metrics)
avg_recall_10 = sum([m["recall@10"] for m in per_query_metrics.values()]) / len(per_query_metrics)
avg_map_10 = sum([m["map@10"] for m in per_query_metrics.values()]) / len(per_query_metrics)

# Print comparison
print("\n" + "="*80)
print("METRICS COMPARISON @ k=10")
print("="*80)
print(f"{'Metric':<20} {'Original':<20} {'Custom (Avg)':<20} {'Difference':<20}")
print("-"*80)
print(f"{'NDCG@10':<20} {ndcg['NDCG@10']:<20.4f} {avg_ndcg_10:<20.4f} {abs(ndcg['NDCG@10'] - avg_ndcg_10):<20.6f}")
print(f"{'Recall@10':<20} {recall['Recall@10']:<20.4f} {avg_recall_10:<20.4f} {abs(recall['Recall@10'] - avg_recall_10):<20.6f}")
print(f"{'MAP@10':<20} {_map['MAP@10']:<20.4f} {avg_map_10:<20.4f} {abs(_map['MAP@10'] - avg_map_10):<20.6f}")
print("="*80)

# Save to JSON file
output_filename = f"{dataset}_per_query_metrics_instructor_paraphrased.json"
with open(output_filename, 'w', encoding='utf-8') as f:
    json.dump(per_query_metrics, f, indent=2, ensure_ascii=False)

print(f"\nPer-query metrics saved to {output_filename}")
print(f"Total queries processed: {len(per_query_metrics)}")


METRICS COMPARISON @ k=10
Metric               Original             Custom (Avg)         Difference          
--------------------------------------------------------------------------------
NDCG@10              0.2904               0.2904               0.000004            
Recall@10            0.1386               0.1386               0.000001            
MAP@10               0.0955               0.0956               0.000004            

Per-query metrics saved to nfcorpus_per_query_metrics_instructor_paraphrased.json
Total queries processed: 323


In [47]:
# Download and load dataset
dataset = "scifact" # dataset name
url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{dataset}.zip"
out_dir = "datasets"
data_path = util.download_and_unzip(url, out_dir)

2025-10-26 23:07:38 - Downloading scifact.zip ...


datasets/scifact.zip:   0%|          | 0.00/2.69M [00:00<?, ?iB/s]

2025-10-26 23:07:46 - Unzipping scifact.zip ...


In [48]:
# Load corpus, queries, and qrels
corpus, queries, qrels = GenericDataLoader(data_folder=data_path).load(split="test")

2025-10-26 23:07:48 - Loading Corpus...


  0%|          | 0/5183 [00:00<?, ?it/s]

2025-10-26 23:07:48 - Loaded 5183 TEST Documents.
2025-10-26 23:07:48 - Doc Example: {'text': 'Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities. A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three-dimensional fiber architecture in cerebral white matter in preterm (n = 17) and full-term infants (n = 7). To assess effects of prematurity on cerebral white matter development, early gestation preterm infants (n = 10) were studied a second time at term. In the central white matter the mean apparent diffusion coefficient at 28 wk was high, 1.8 microm2/ms, and decreased toward term to 1.2 microm2/ms. In the posterior limb of the internal capsule, the mean apparent diffusion coefficients at both times were similar (1.2 vers

In [49]:
import json
# load paraphrased dataset
with open(f"{dataset}_query_paraphrased_gpt4o.json", encoding='utf-8') as f:
    # Load the JSON data into a Python dictionary
    queries_para = json.load(f)

In [50]:
queries_p = {}
for q in queries_para:
  queries_p[q] = queries_para[q]['query_p']

In [51]:
results_instructor = retriever_instructor.retrieve(corpus, queries_p)

2025-10-26 23:08:22 - Encoding Queries...


  0%|          | 0/3 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

 33%|███▎      | 1/3 [00:00<00:00,  8.36it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

 95%|█████████▌| 39/41 [00:27<00:00,  2.31it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

 98%|█████████▊| 40/41 [00:27<00:00,  2.50it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 41/41 [00:27<00:00,  1.47it/s]


In [52]:
# Evaluate your model with NDCG@k, MAP@K, Recall@K and Precision@K where k = [1,3,5,10,100,1000]
ndcg, _map, recall, precision = retriever_instructor.evaluate(qrels, results_instructor, retriever_instructor.k_values)

2025-10-26 23:09:51 - For evaluation, we ignore identical query and document ids (default), please explicitly set ``ignore_identical_ids=False`` to ignore this.
2025-10-26 23:09:51 - 

2025-10-26 23:09:51 - NDCG@1: 0.5133
2025-10-26 23:09:51 - NDCG@3: 0.5708
2025-10-26 23:09:51 - NDCG@5: 0.6012
2025-10-26 23:09:51 - NDCG@10: 0.6236
2025-10-26 23:09:51 - NDCG@100: 0.6513
2025-10-26 23:09:51 - NDCG@1000: 0.6655
2025-10-26 23:09:51 - 

2025-10-26 23:09:51 - MAP@1: 0.4892
2025-10-26 23:09:51 - MAP@3: 0.5454
2025-10-26 23:09:51 - MAP@5: 0.5654
2025-10-26 23:09:51 - MAP@10: 0.5759
2025-10-26 23:09:51 - MAP@100: 0.5828
2025-10-26 23:09:51 - MAP@1000: 0.5834
2025-10-26 23:09:51 - 

2025-10-26 23:09:51 - Recall@1: 0.4892
2025-10-26 23:09:51 - Recall@3: 0.6151
2025-10-26 23:09:51 - Recall@5: 0.6916
2025-10-26 23:09:51 - Recall@10: 0.7551
2025-10-26 23:09:51 - Recall@100: 0.8750
2025-10-26 23:09:51 - Recall@1000: 0.9867
2025-10-26 23:09:51 - 

2025-10-26 23:09:51 - P@1: 0.5133
2025-10-26 23:09:51

In [53]:
print(f"Model: Instructor; Dataset: {dataset} (paraphrased)")
print("-" * 150)
print(ndcg)
print(_map)
print(recall)
print(precision)

Model: Instructor; Dataset: scifact (paraphrased)
------------------------------------------------------------------------------------------------------------------------------------------------------
{'NDCG@1': 0.51333, 'NDCG@3': 0.57079, 'NDCG@5': 0.60121, 'NDCG@10': 0.6236, 'NDCG@100': 0.65127, 'NDCG@1000': 0.6655}
{'MAP@1': 0.48917, 'MAP@3': 0.54537, 'MAP@5': 0.56537, 'MAP@10': 0.57593, 'MAP@100': 0.58285, 'MAP@1000': 0.58337}
{'Recall@1': 0.48917, 'Recall@3': 0.61506, 'Recall@5': 0.69156, 'Recall@10': 0.75511, 'Recall@100': 0.875, 'Recall@1000': 0.98667}
{'P@1': 0.51333, 'P@3': 0.22556, 'P@5': 0.15267, 'P@10': 0.08433, 'P@100': 0.00997, 'P@1000': 0.00112}


In [54]:
import json
import pytrec_eval

# Use pytrec_eval to get per-query metrics
k_values = [10]
map_string = "map_cut." + ",".join([str(k) for k in k_values])
ndcg_string = "ndcg_cut." + ",".join([str(k) for k in k_values])
recall_string = "recall." + ",".join([str(k) for k in k_values])
precision_string = "P." + ",".join([str(k) for k in k_values])

evaluator = pytrec_eval.RelevanceEvaluator(qrels, {map_string, ndcg_string, recall_string, precision_string})
scores = evaluator.evaluate(results_instructor)

# Extract per-query metrics
per_query_metrics = {}
for query_id in scores.keys():
    per_query_metrics[query_id] = {
        "ndcg@10": scores[query_id]["ndcg_cut_10"],
        "recall@10": scores[query_id]["recall_10"],
        "map@10": scores[query_id]["map_cut_10"]
    }

# Calculate average metrics
avg_ndcg_10 = sum([m["ndcg@10"] for m in per_query_metrics.values()]) / len(per_query_metrics)
avg_recall_10 = sum([m["recall@10"] for m in per_query_metrics.values()]) / len(per_query_metrics)
avg_map_10 = sum([m["map@10"] for m in per_query_metrics.values()]) / len(per_query_metrics)

# Print comparison
print("\n" + "="*80)
print("METRICS COMPARISON @ k=10")
print("="*80)
print(f"{'Metric':<20} {'Original':<20} {'Custom (Avg)':<20} {'Difference':<20}")
print("-"*80)
print(f"{'NDCG@10':<20} {ndcg['NDCG@10']:<20.4f} {avg_ndcg_10:<20.4f} {abs(ndcg['NDCG@10'] - avg_ndcg_10):<20.6f}")
print(f"{'Recall@10':<20} {recall['Recall@10']:<20.4f} {avg_recall_10:<20.4f} {abs(recall['Recall@10'] - avg_recall_10):<20.6f}")
print(f"{'MAP@10':<20} {_map['MAP@10']:<20.4f} {avg_map_10:<20.4f} {abs(_map['MAP@10'] - avg_map_10):<20.6f}")
print("="*80)

# Save to JSON file
output_filename = f"{dataset}_per_query_metrics_instructor_paraphrased.json"
with open(output_filename, 'w', encoding='utf-8') as f:
    json.dump(per_query_metrics, f, indent=2, ensure_ascii=False)

print(f"\nPer-query metrics saved to {output_filename}")
print(f"Total queries processed: {len(per_query_metrics)}")


METRICS COMPARISON @ k=10
Metric               Original             Custom (Avg)         Difference          
--------------------------------------------------------------------------------
NDCG@10              0.6236               0.6236               0.000001            
Recall@10            0.7551               0.7551               0.000001            
MAP@10               0.5759               0.5759               0.000003            

Per-query metrics saved to scifact_per_query_metrics_instructor_paraphrased.json
Total queries processed: 300


In [55]:
# Download and load dataset
dataset = "trec-covid" # dataset name
url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{dataset}.zip"
out_dir = "datasets"
data_path = util.download_and_unzip(url, out_dir)

2025-10-26 23:18:31 - Downloading trec-covid.zip ...


datasets/trec-covid.zip:   0%|          | 0.00/70.5M [00:00<?, ?iB/s]

2025-10-26 23:19:02 - Unzipping trec-covid.zip ...


In [56]:
# Load corpus, queries, and qrels
corpus, queries, qrels = GenericDataLoader(data_folder=data_path).load(split="test")

2025-10-26 23:19:03 - Loading Corpus...


  0%|          | 0/171332 [00:00<?, ?it/s]

2025-10-26 23:19:04 - Loaded 171332 TEST Documents.
2025-10-26 23:19:04 - Doc Example: {'text': 'OBJECTIVE: This retrospective chart review describes the epidemiology and clinical features of 40 patients with culture-proven Mycoplasma pneumoniae infections at King Abdulaziz University Hospital, Jeddah, Saudi Arabia. METHODS: Patients with positive M. pneumoniae cultures from respiratory specimens from January 1997 through December 1998 were identified through the Microbiology records. Charts of patients were reviewed. RESULTS: 40 patients were identified, 33 (82.5%) of whom required admission. Most infections (92.5%) were community-acquired. The infection affected all age groups but was most common in infants (32.5%) and pre-school children (22.5%). It occurred year-round but was most common in the fall (35%) and spring (30%). More than three-quarters of patients (77.5%) had comorbidities. Twenty-four isolates (60%) were associated with pneumonia, 14 (35%) with upper respiratory tract 

In [57]:
import json
# load paraphrased dataset
with open(f"{dataset}_query_paraphrased_gpt4o.json", encoding='utf-8') as f:
    # Load the JSON data into a Python dictionary
    queries_para = json.load(f)

In [58]:
queries_p = {}
for q in queries_para:
  queries_p[q] = queries_para[q]['query_p']

In [59]:
results_instructor = retriever_instructor.retrieve(corpus, queries_p)

2025-10-26 23:19:04 - Encoding Queries...


  0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00, 23.52it/s]

2025-10-26 23:19:04 - Sorting Corpus by document length (Longest first)...


...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

 99%|█████████▉| 165/167 [00:08<00:00, 28.29it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 167/167 [00:08<00:00, 19.53it/s]


In [63]:
# Evaluate your model with NDCG@k, MAP@K, Recall@K and Precision@K where k = [1,3,5,10,100,1000]
ndcg, _map, recall, precision = retriever_instructor.evaluate(qrels, results_instructor, retriever_instructor.k_values)

2025-10-26 23:39:03 - For evaluation, we ignore identical query and document ids (default), please explicitly set ``ignore_identical_ids=False`` to ignore this.
2025-10-26 23:39:03 - 

2025-10-26 23:39:03 - NDCG@1: 0.6000
2025-10-26 23:39:03 - NDCG@3: 0.5004
2025-10-26 23:39:03 - NDCG@5: 0.4704
2025-10-26 23:39:03 - NDCG@10: 0.4362
2025-10-26 23:39:03 - NDCG@100: 0.3285
2025-10-26 23:39:03 - NDCG@1000: 0.3398
2025-10-26 23:39:03 - 

2025-10-26 23:39:03 - MAP@1: 0.0016
2025-10-26 23:39:03 - MAP@3: 0.0034
2025-10-26 23:39:03 - MAP@5: 0.0049
2025-10-26 23:39:03 - MAP@10: 0.0085
2025-10-26 23:39:03 - MAP@100: 0.0470
2025-10-26 23:39:03 - MAP@1000: 0.1368
2025-10-26 23:39:03 - 

2025-10-26 23:39:03 - Recall@1: 0.0016
2025-10-26 23:39:03 - Recall@3: 0.0038
2025-10-26 23:39:03 - Recall@5: 0.0058
2025-10-26 23:39:03 - Recall@10: 0.0112
2025-10-26 23:39:03 - Recall@100: 0.0772
2025-10-26 23:39:03 - Recall@1000: 0.3342
2025-10-26 23:39:03 - 

2025-10-26 23:39:03 - P@1: 0.6600
2025-10-26 23:39:03

In [64]:
print(f"Model: Instructor; Dataset: {dataset} (paraphrased)")
print("-" * 150)
print(ndcg)
print(_map)
print(recall)
print(precision)

Model: Instructor; Dataset: trec-covid (paraphrased)
------------------------------------------------------------------------------------------------------------------------------------------------------
{'NDCG@1': 0.6, 'NDCG@3': 0.50039, 'NDCG@5': 0.47038, 'NDCG@10': 0.43617, 'NDCG@100': 0.32846, 'NDCG@1000': 0.33977}
{'MAP@1': 0.00158, 'MAP@3': 0.00344, 'MAP@5': 0.0049, 'MAP@10': 0.00847, 'MAP@100': 0.04704, 'MAP@1000': 0.13678}
{'Recall@1': 0.00158, 'Recall@3': 0.00378, 'Recall@5': 0.00579, 'Recall@10': 0.01118, 'Recall@100': 0.07721, 'Recall@1000': 0.33422}
{'P@1': 0.66, 'P@3': 0.53333, 'P@5': 0.492, 'P@10': 0.456, 'P@100': 0.3358, 'P@1000': 0.15974}


In [65]:
import json
import pytrec_eval

# Debug: Check if all queries are present
print(f"Number of queries in qrels: {len(qrels)}")
print(f"Number of queries in results_instructor: {len(results_instructor)}")
print(f"Missing queries: {set(qrels.keys()) - set(results_instructor.keys())}")

# Filter out identical query and document IDs (same as BEIR does by default)
results_filtered = {}
for qid, rels in results_instructor.items():
    results_filtered[qid] = {}
    for pid, score in rels.items():
        if qid != pid:  # Skip if query ID matches document ID
            results_filtered[qid][pid] = score

# Use pytrec_eval to get per-query metrics
k_values = [10]
map_string = "map_cut." + ",".join([str(k) for k in k_values])
ndcg_string = "ndcg_cut." + ",".join([str(k) for k in k_values])
recall_string = "recall." + ",".join([str(k) for k in k_values])
precision_string = "P." + ",".join([str(k) for k in k_values])

evaluator = pytrec_eval.RelevanceEvaluator(qrels, {map_string, ndcg_string, recall_string, precision_string})
scores = evaluator.evaluate(results_filtered)

print(f"Number of queries in scores: {len(scores)}")
print(f"Missing from scores: {set(qrels.keys()) - set(scores.keys())}")

# Extract per-query metrics
per_query_metrics = {}
for query_id in scores.keys():
    per_query_metrics[query_id] = {
        "ndcg@10": scores[query_id]["ndcg_cut_10"],
        "recall@10": scores[query_id]["recall_10"],
        "map@10": scores[query_id]["map_cut_10"]
    }

# Calculate average metrics
avg_ndcg_10 = sum([m["ndcg@10"] for m in per_query_metrics.values()]) / len(per_query_metrics)
avg_recall_10 = sum([m["recall@10"] for m in per_query_metrics.values()]) / len(per_query_metrics)
avg_map_10 = sum([m["map@10"] for m in per_query_metrics.values()]) / len(per_query_metrics)

# Print comparison
print("\n" + "="*80)
print("METRICS COMPARISON @ k=10")
print("="*80)
print(f"{'Metric':<20} {'Original':<20} {'Custom (Avg)':<20} {'Difference':<20}")
print("-"*80)
print(f"{'NDCG@10':<20} {ndcg['NDCG@10']:<20.4f} {avg_ndcg_10:<20.4f} {abs(ndcg['NDCG@10'] - avg_ndcg_10):<20.6f}")
print(f"{'Recall@10':<20} {recall['Recall@10']:<20.4f} {avg_recall_10:<20.4f} {abs(recall['Recall@10'] - avg_recall_10):<20.6f}")
print(f"{'MAP@10':<20} {_map['MAP@10']:<20.4f} {avg_map_10:<20.4f} {abs(_map['MAP@10'] - avg_map_10):<20.6f}")
print("="*80)

# Save to JSON file
output_filename = f"{dataset}_per_query_metrics_instructor_paraphrased.json"
with open(output_filename, 'w', encoding='utf-8') as f:
    json.dump(per_query_metrics, f, indent=2, ensure_ascii=False)

print(f"\nPer-query metrics saved to {output_filename}")
print(f"Total queries processed: {len(per_query_metrics)}")

Number of queries in qrels: 50
Number of queries in results_instructor: 50
Missing queries: set()
Number of queries in scores: 50
Missing from scores: set()

METRICS COMPARISON @ k=10
Metric               Original             Custom (Avg)         Difference          
--------------------------------------------------------------------------------
NDCG@10              0.4362               0.4362               0.000001            
Recall@10            0.0112               0.0112               0.000000            
MAP@10               0.0085               0.0085               0.000001            

Per-query metrics saved to trec-covid_per_query_metrics_instructor_paraphrased.json
Total queries processed: 50
